# AICE Associate 모의고사 25문항 답안지

- 데이터셋: Kaggle Titanic `train.csv`
- 전제: 이 노트북 파일과 같은 폴더에 `train.csv`가 있어야 합니다.
- 주요 범위: 데이터 로드, 결측치 처리, 시각화, 인코딩, 스케일링, 모델링, 평가, 그룹 분석

## 공통 라이브러리 import

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import warnings
warnings.filterwarnings('ignore')

## 문제 1. Titanic 데이터를 불러와 `df` 변수에 저장하시오.

In [ ]:
df = pd.read_csv('train.csv')
df.head()

## 문제 2. 데이터의 행과 열 개수를 확인하시오.

In [ ]:
df.shape

## 문제 3. 각 컬럼의 결측치 개수를 확인하시오.

In [ ]:
df.isnull().sum()

## 문제 4. `Age` 컬럼의 결측치를 평균값으로 대체하시오.

In [ ]:
df['Age'] = df['Age'].fillna(df['Age'].mean())

df['Age'].isnull().sum()

## 문제 5. `Embarked` 컬럼의 결측치를 최빈값으로 대체하시오.

In [ ]:
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

df['Embarked'].isnull().sum()

## 문제 6. `Survived` 컬럼의 분포를 막대그래프로 시각화하시오.

In [ ]:
sns.countplot(data=df, x='Survived')
plt.title('Survived Distribution')
plt.xlabel('Survived')
plt.ylabel('Count')
plt.show()

## 문제 7. `Age` 컬럼의 분포를 히스토그램으로 시각화하시오.

In [ ]:
sns.histplot(df['Age'], bins=30)
plt.title('Age Distribution')
plt.xlabel('Age')
plt.ylabel('Count')
plt.show()

## 문제 8. `Sex`, `Embarked`를 One-Hot Encoding 하시오. 첫 번째 카테고리는 제거할 것.

In [ ]:
df_encoded = pd.get_dummies(
    df,
    columns=['Sex', 'Embarked'],
    drop_first=True
)

df_encoded.head()

## 문제 9. 지정된 컬럼으로 Feature(X)를 생성하고, `Survived`를 Target(y)으로 생성하시오.

In [ ]:
feature_cols = [
    'Pclass',
    'Age',
    'Fare',
    'Sex_male',
    'Embarked_Q',
    'Embarked_S'
]

X = df_encoded[feature_cols]
y = df_encoded['Survived']

print(X.shape)
print(y.shape)

## 문제 10. 학습 데이터와 테스트 데이터를 분리하시오. `test_size=0.2`, `random_state=42`

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape, X_test.shape)
print(y_train.shape, y_test.shape)

## 문제 11. `StandardScaler`를 사용하여 학습 데이터와 테스트 데이터를 스케일링하시오.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled[:5]

## 문제 12. Random Forest 모델을 생성하시오. `n_estimators=100`, `random_state=42`

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model

## 문제 13. 모델을 학습시키고 테스트 데이터에 대한 예측값을 생성하시오.

In [ ]:
rf_model.fit(X_train_scaled, y_train)

rf_pred = rf_model.predict(X_test_scaled)

rf_pred[:10]

## 문제 14. Accuracy, Confusion Matrix, Classification Report를 출력하시오.

In [ ]:
print('Accuracy:', accuracy_score(y_test, rf_pred))

print('\nConfusion Matrix')
print(confusion_matrix(y_test, rf_pred))

print('\nClassification Report')
print(classification_report(y_test, rf_pred))

## 문제 15. 중복 데이터 개수를 확인하시오.

In [ ]:
df.duplicated().sum()

## 문제 16. 중복 데이터를 제거한 새로운 데이터프레임을 생성하시오.

In [ ]:
df_no_dup = df.drop_duplicates()

print('Before:', df.shape)
print('After:', df_no_dup.shape)

## 문제 17. 숫자형 컬럼들에 대한 상관관계 행렬을 구하시오.

In [ ]:
corr_matrix = df.corr(numeric_only=True)

corr_matrix

## 문제 18. 상관관계 행렬을 Heatmap으로 시각화하시오. 상관계수 값을 표시할 것.

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f'
)
plt.title('Correlation Heatmap')
plt.show()

## 문제 19. Random Forest 모델의 Feature Importance를 출력하고 중요도가 높은 순으로 정렬하시오.

In [ ]:
feature_importance = pd.Series(
    rf_model.feature_importances_,
    index=feature_cols
).sort_values(ascending=False)

feature_importance

## 문제 20. `Age`와 `Fare` 컬럼에 대해 Boxplot을 그려 이상치를 확인하시오.

In [ ]:
plt.figure(figsize=(6, 4))
sns.boxplot(y=df['Age'])
plt.title('Age Boxplot')
plt.show()

plt.figure(figsize=(6, 4))
sns.boxplot(y=df['Fare'])
plt.title('Fare Boxplot')
plt.show()

## 문제 21. `Age` 컬럼을 기준으로 `Age_Group` 컬럼을 생성하시오.

In [ ]:
def age_group(age):
    if age <= 19:
        return 'Young'
    elif age <= 59:
        return 'Adult'
    else:
        return 'Senior'

df['Age_Group'] = df['Age'].apply(age_group)

df[['Age', 'Age_Group']].head(10)

## 문제 22. 생성한 `Age_Group`별 생존율을 계산하시오.

In [ ]:
age_group_survival = df.groupby('Age_Group')['Survived'].mean()

age_group_survival

## 문제 23. 성별(`Sex`)에 따른 생존율을 계산하시오.

In [ ]:
sex_survival = df.groupby('Sex')['Survived'].mean()

sex_survival

## 문제 24. 객실등급(`Pclass`)별 평균 운임(`Fare`)을 계산하시오.

In [ ]:
pclass_fare_mean = df.groupby('Pclass')['Fare'].mean()

pclass_fare_mean

## 문제 25. 생존 여부(`Survived`)를 예측하는 Logistic Regression 모델을 생성하고 성능을 평가하시오.

In [ ]:
log_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

log_model.fit(X_train_scaled, y_train)

log_pred = log_model.predict(X_test_scaled)

print('Accuracy:', accuracy_score(y_test, log_pred))

print('\nConfusion Matrix')
print(confusion_matrix(y_test, log_pred))

print('\nClassification Report')
print(classification_report(y_test, log_pred))